## 00. Quick Start


In [1]:
print('Concept Portfolio V2 Lab — 기본 MODE는 MOCK입니다.')
print('LIVE는 06→28 순서의 staged 검증 후 46의 가드를 직접 켜야 합니다.')

Concept Portfolio V2 Lab — 기본 MODE는 MOCK입니다.
LIVE는 06→28 순서의 staged 검증 후 46의 가드를 직접 켜야 합니다.


## 01. Environment


In [2]:
import os, sys, json
from pathlib import Path
from IPython.display import display
SEARCH_ROOTS = [Path.cwd(), *Path.cwd().parents]
AI_ROOT = next((p for p in SEARCH_ROOTS if (p / 'app').is_dir()), None)
if AI_ROOT is None: AI_ROOT = next((p / 'ai' for p in SEARCH_ROOTS if (p / 'ai' / 'app').is_dir()))
if str(AI_ROOT) not in sys.path: sys.path.insert(0, str(AI_ROOT))
print({'python': sys.version.split()[0], 'aiRoot': str(AI_ROOT)})

{'python': '3.14.5', 'aiRoot': 'C:\\Users\\seewo\\Desktop\\big_proj_01\\new_3\\ai'}


## 02. MODE


In [3]:
MODE = 'LIVE'  # MOCK | REPLAY | LIVE
RECORDINGS_DIR = AI_ROOT / 'recordings' / 'concept_portfolio_v2'
print({'mode': MODE, 'liveExternalOperationsEnabled': MODE == 'LIVE'})

{'mode': 'LIVE', 'liveExternalOperationsEnabled': True}


## 03. Environment Check


In [4]:
LIVE_ENV_KEYS = ['AI_PROVIDER', 'AI_API_KEY', 'AI_MODEL', 'MOLEG_API_KEY', 'LEGAL_REGISTRY_VERSION']
env_status = {key: bool(os.getenv(key)) for key in LIVE_ENV_KEYS}
print(env_status if MODE == 'LIVE' else {'mode': MODE, 'message': '외부 환경변수 불필요'})

{'AI_PROVIDER': True, 'AI_API_KEY': True, 'AI_MODEL': True, 'MOLEG_API_KEY': True, 'LEGAL_REGISTRY_VERSION': True}


## 04. Schema Preflight


In [5]:
from app.concept_portfolio_v2 import ConceptPortfolioEngine, ProviderGateway, ProviderMode
from app.concept_portfolio_v2.adapters import CurrentLegalAdapter
from app.concept_portfolio_v2.diagnostics.notebook_view import *
gateway = ProviderGateway(MODE, recordings_dir=RECORDINGS_DIR)
engine = ConceptPortfolioEngine(MODE, gateway=gateway)
schema_preflight = engine.schema_preflight_report()
display(show_schema_preflight(schema_preflight))
assert schema_preflight.status == 'PASS' and schema_preflight.providerCalls == 0

,스키마,상태,실패,Provider 호출
0,PlanDraftPool,PASS,[],0
1,ConceptCandidateDraft,PASS,[],0
2,SemanticDistinctnessResult,PASS,[],0
3,SemanticFidelityResult,PASS,[],0
4,SemanticArchitectureBatch,PASS,[],0


## 05. Input


In [6]:
TEST_INPUT = {
    'ideaOverview': '개인 맞춤형 소량 식재료를 제공하고 남은 재료 활용 레시피를 안내하는 서비스',
    'problem': '1~2인 가구가 큰 포장 단위 때문에 식재료를 남기고 음식물 쓰레기가 발생한다',
    'targetUsers': '요리할 시간이 적고 낭비를 줄이고 싶은 1~2인 가구',
}
MAX_CONCEPTS = 5

## 06. Idea Brief Derivation


In [7]:
engine._reset()
seed = engine.seed_adapter.adapt(TEST_INPUT)
idea_context = await engine.derive_idea_brief(seed)
print({'ideaCallComplete': True, 'interpretationPresent': bool(seed.interpretation)})

{'ideaCallComplete': True, 'interpretationPresent': True}


## 07. Safety


In [8]:
display(idea_context.safetyReview.model_dump(mode='json'))
assert idea_context.safetyReview.passed

{'decision': 'ALLOW',
 'categories': [],
 'restrictions': [],
 'userFacingReason': '이 아이디어는 안전하며, 개인 맞춤형 식재료 제공과 관련된 문제를 해결하는 데 기여할 수 있습니다.'}

## 08. AI가 이해한 아이디어


In [9]:
display(show_idea_interpretation(idea_context))

,항목,AI 이해 결과
0,interpretedProblem,1~2인 가구에서 발생하는 음식물 쓰레기 문제를 해결하기 위한 서비스.
1,interpretedTargetUsers,요리할 시간이 적고 식재료 낭비를 줄이고 싶은 1~2인 가구.
2,usageContext,소량의 식재료를 필요로 하는 소비자에게 맞춤형 서비스를 제공하는 상황.
3,industryCategory,식품 및 요리 서비스
4,researchScope,소량 식재료 제공 및 레시피 안내 서비스에 대한 시장 조사.
5,conciseIdeaDefinition,개인 맞춤형 소량 식재료와 활용 레시피를 제공하는 서비스.
6,targetRegionInterpretation,특정 지역에 대한 정보가 필요합니다.
7,relevantKnownCompetitorContext,경쟁자에 대한 정보가 필요합니다.


## 09. Readiness / Summary / commitments


In [10]:
display(show_idea_readiness(idea_context))

{'readiness': {'status': 'READY_FOR_REVIEW',
  'score': 0,
  'missingFieldKeys': []},
 'readinessDiagnostic': 'READINESS_INCONSISTENT',
 'userFacingSummary': '이 서비스는 개인 맞춤형 소량 식재료를 제공하고, 남은 재료를 활용할 수 있는 레시피를 안내하여 1~2인 가구의 음식물 쓰레기 문제를 해결하는 데 도움을 줄 수 있습니다.',
 'commitmentCandidates': [],
 'contradictions': [],
 'questions': []}

## 10. Seed Analysis


In [11]:
analysis = await engine.analyze_seed(seed)
display(show_seed_analysis(analysis))

,구분,값
0,탐색 폭,EXPLORE
1,다양성 수용량,5
2,설명,선택 입력 LOCK 0개로 11개 설계 차원이 열려 있습니다. diversityCa...


## 11. Generic Opportunity Kernel


In [12]:
display(analysis.opportunityKernel.model_dump(mode='json'))

{'problemCore': '1~2인 가구에서 발생하는 음식물 쓰레기 문제를 해결하기 위한 서비스.',
 'targetCore': '요리할 시간이 적고 식재료 낭비를 줄이고 싶은 1~2인 가구.',
 'useContexts': ['소량의 식재료를 필요로 하는 소비자에게 맞춤형 서비스를 제공하는 상황.'],
 'intentComponents': ['개인 맞춤형 소량 식재료와 활용 레시피를 제공하는 서비스.'],
 'mustPreserve': ['1~2인 가구에서 발생하는 음식물 쓰레기 문제를 해결하기 위한 서비스.',
  '요리할 시간이 적고 식재료 낭비를 줄이고 싶은 1~2인 가구.',
  '개인 맞춤형 소량 식재료와 활용 레시피를 제공하는 서비스.'],
 'maySpecialize': ['핵심 대상의 의미 있는 하위 세그먼트',
  '핵심 사용 맥락의 구체화',
  '가치 제안 또는 offer의 구체화'],
 'forbiddenDriftSummary': '핵심 문제와 대상이 모두 무관한 기회로 교체되면 범위를 벗어납니다.'}

## 12. Design Space


In [13]:
display(show_design_space(analysis))

,분류,필드,값
0,SOURCE_LOCK,ideaOverview,개인 맞춤형 소량 식재료를 제공하고 남은 재료 활용 레시피를 안내하는 서비스
1,SOURCE_LOCK,problem,1~2인 가구가 큰 포장 단위 때문에 식재료를 남기고 음식물 쓰레기가 발생한다
2,SOURCE_LOCK,targetUsers,요리할 시간이 적고 낭비를 줄이고 싶은 1~2인 가구
3,SEMANTIC_ANCHOR,ideaOverview,개인 맞춤형 소량 식재료를 제공하고 남은 재료 활용 레시피를 안내하는 서비스
4,SEMANTIC_ANCHOR,problem,1~2인 가구가 큰 포장 단위 때문에 식재료를 남기고 음식물 쓰레기가 발생한다
5,SEMANTIC_ANCHOR,targetUsers,요리할 시간이 적고 낭비를 줄이고 싶은 1~2인 가구
6,OPEN,solutionMechanism,변경 가능
7,OPEN,valueDelivery,변경 가능
8,OPEN,operatingModel,변경 가능
9,OPEN,supplyStructure,변경 가능


## 13. Generate and Adaptively Replenish Plan Pool


In [14]:
plan_validation = await engine.prepare_portfolio_plans(seed, analysis, max_concepts=MAX_CONCEPTS)
plans = engine._last_plan_pool
print({'totalPlans': len(plans), 'planningRounds': plan_validation.planningRounds,
       'replenishmentRequested': plan_validation.replenishmentRequested})

{'totalPlans': 7, 'planningRounds': 1, 'replenishmentRequested': 0}


## 14. Plan Count / Adaptive Replenishment Check


In [15]:
display(show_plan_pool_status(engine._last_plan_pool_status))
display({'planningRounds': plan_validation.planningRounds,
         'replenishmentRequested': plan_validation.replenishmentRequested,
         'adaptiveReplenishmentUsed': plan_validation.planningRounds > 1})

,requestedPoolSize,returnedPoolSize,initialTarget,reserveTarget,reserveAvailable,status
0,7,7,5,2,2,RESERVE_READY


{'planningRounds': 1,
 'replenishmentRequested': 0,
 'adaptiveReplenishmentUsed': False}

## 15. Korean Plan Display


In [16]:
display(show_portfolio_plans(plan_validation.acceptedPlans + plan_validation.reservePlans))

,planId,제목,선택 상태,selectionScore,selectionReason,relationToPortfolio,Concept Family,Target Thesis,Use Context,Value Thesis,Offer Thesis,Solution Thesis,Architecture,비교 가치
0,P7,요리 초보를 위한 맞춤형 서비스,SELECTED,0.8373,Opportunity fit과 Concept clarity가 가장 높은 대표안,PORTFOLIO_SEED,기타 역할 · 기타 운영,요리할 시간이 적고 식재료 낭비를 줄이고 싶은 1~2인 가구,소량의 식재료를 필요로 하는 소비자에게 맞춤형 서비스를 제공하는 상황,요리 초보를 위한 맞춤형 식재료와 레시피를 제공하여 요리의 즐거움을 더한다.,요리 초보를 위한 맞춤형 식재료와 레시피 제공으로 요리의 즐거움을 더한다.,요리 초보를 위한 소량 식재료와 간편한 레시피를 통해 음식물 쓰레기를 줄인다.,"{'businessRole': 'OTHER', 'operatingModel': 'O...",요리 초보의 문제를 해결하기 위한 핵심 서비스.
1,P1,소량 맞춤형 식재료 서비스,SELECTED,0.8302,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,기타 역할 · 기타 운영,요리할 시간이 적고 식재료 낭비를 줄이고 싶은 1~2인 가구,소량의 식재료를 필요로 하는 소비자에게 맞춤형 서비스를 제공하는 상황,필요한 만큼의 식재료를 제공하여 음식물 쓰레기를 줄이고 요리를 쉽게 할 수 있도록 ...,소량의 식재료와 레시피를 제공하여 1~2인 가구의 요리 경험을 개선한다.,"개인 맞춤형 식재료와 레시피를 통해 요리의 번거로움을 줄이고, 음식물 쓰레기를 최소...","{'businessRole': 'OTHER', 'operatingModel': 'O...",1~2인 가구의 요리 문제를 해결하기 위한 핵심 서비스.
2,P3,간편 요리 키트 서비스,SELECTED,0.7430,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,기타 역할 · 기타 운영,요리할 시간이 적고 식재료 낭비를 줄이고 싶은 1~2인 가구,소량의 식재료를 필요로 하는 소비자에게 맞춤형 서비스를 제공하는 상황,간편하게 요리할 수 있는 키트를 제공하여 요리의 번거로움을 줄인다.,간편 요리 키트를 통해 요리의 즐거움을 제공한다.,소량의 식재료와 간편한 레시피를 통해 요리의 접근성을 높인다.,"{'businessRole': 'OTHER', 'operatingModel': 'O...",요리의 번거로움을 줄이고 1~2인 가구의 문제를 해결하는 서비스.
3,P4,맞춤형 식사 계획 서비스,SELECTED,0.6346,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,기타 역할 · 기타 운영,요리할 시간이 적고 식재료 낭비를 줄이고 싶은 1~2인 가구,소량의 식재료를 필요로 하는 소비자에게 맞춤형 서비스를 제공하는 상황,개인 맞춤형 식사 계획을 통해 요리의 효율성을 높인다.,맞춤형 식사 계획과 소량 식재료 제공으로 요리의 편리함을 더한다.,고객의 식사 스타일에 맞춘 식사 계획과 식재료 제공으로 음식물 쓰레기를 줄인다.,"{'businessRole': 'OTHER', 'operatingModel': 'O...",1~2인 가구의 식사 문제를 해결하기 위한 핵심 서비스.
4,P5,건강한 소량 식재료 서비스,SELECTED,0.5475,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,기타 역할 · 기타 운영,요리할 시간이 적고 식재료 낭비를 줄이고 싶은 1~2인 가구,소량의 식재료를 필요로 하는 소비자에게 맞춤형 서비스를 제공하는 상황,건강한 소량 식재료를 제공하여 요리의 질을 높인다.,건강한 식재료를 소량으로 제공하여 요리의 질을 높인다.,소량의 건강한 식재료를 통해 음식물 쓰레기를 줄인다.,"{'businessRole': 'OTHER', 'operatingModel': 'O...",건강한 식재료를 제공하여 1~2인 가구의 문제를 해결하는 서비스.
5,P2,레시피 기반 식재료 패키지,RESERVE,0.6939,Selected Portfolio 대비 marginal value가 낮아 reser...,"DISTINCT,VARIANT",기타 역할 · 기타 운영,요리할 시간이 적고 식재료 낭비를 줄이고 싶은 1~2인 가구,소량의 식재료를 필요로 하는 소비자에게 맞춤형 서비스를 제공하는 상황,레시피에 맞춘 소량의 식재료를 제공하여 요리의 편리함을 더한다.,레시피에 최적화된 식재료 패키지를 제공하여 요리의 효율성을 높인다.,고객의 요리 스타일에 맞춘 식재료 패키지를 통해 음식물 쓰레기를 줄인다.,"{'businessRole': 'OTHER', 'operatingModel': 'O...",요리의 편리함을 제공하여 1~2인 가구의 문제를 해결하는 서비스.
6,P6,신선한 재료 정기 배송 서비스,RESERVE,0.6825,Selected Portfolio 대비 marginal value가 낮아 reser...,"DISTINCT,VARIANT",기타 역할 · 기타 운영,요리할 시간이 적고 식재료 낭비를 줄이고 싶은 1~2인 가구,소량의 식재료를 필요로 하는 소비자에게 맞춤형 서비스를 제공하는 상황,신선한 식재료를 정기적으로 제공하여 요리의 질을 높인다.,신선한 식재료를 정기적으로 제공하여 요리의 질을 높인다.,정기 배송을 통해 신선한 식재료를 제공하여 음식물 쓰레기를 줄인다.,"{'businessRole': 'OTHER', 'operatingModel': 'O...",신선한 식재료를 제공하여 1~2인 가구의 문제를 해결하는 서비스.


## 16. Plan Lock/Intent Validation


In [17]:
display({'accepted': [p.planId for p in plan_validation.acceptedPlans],
         'rejected': [p.model_dump(mode='json') for p in plan_validation.rejectedPlans]})

{'accepted': ['P7', 'P1', 'P3', 'P4', 'P5'], 'rejected': []}

## 17. Portfolio Family / Variant / Distinct


In [18]:
display(show_plan_diversity(plan_validation.diversity))

,A,B,판정,Family A,Family B,겹침,실질 차이,단계,semantic judge,관계 설명
0,P1,P2,DISTINCT,OTHER:OTHER,OTHER:OTHER,"businessRole, operatingModel, partnerModel, de...","customerInteractionModel, valuePropositionThes...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
1,P1,P3,DISTINCT,OTHER:OTHER,OTHER:OTHER,"businessRole, operatingModel, deliveryModel, t...","partnerModel, customerInteractionModel, valueP...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
2,P2,P3,DISTINCT,OTHER:OTHER,OTHER:OTHER,"businessRole, operatingModel, deliveryModel, t...","partnerModel, valuePropositionThesis, offerThe...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
3,P1,P4,DISTINCT,OTHER:OTHER,OTHER:OTHER,"businessRole, operatingModel, deliveryModel, t...","partnerModel, customerInteractionModel, valueP...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
4,P2,P4,VARIANT,OTHER:OTHER,OTHER:OTHER,"businessRole, operatingModel, deliveryModel, t...","partnerModel, valuePropositionThesis, offerThe...",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...
5,P3,P4,DISTINCT,OTHER:OTHER,OTHER:OTHER,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, offerThesis, solutionT...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
6,P1,P5,DISTINCT,OTHER:OTHER,OTHER:OTHER,"businessRole, operatingModel, partnerModel, de...","customerInteractionModel, valuePropositionThes...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
7,P2,P5,VARIANT,OTHER:OTHER,OTHER:OTHER,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, offerThesis, solutionT...",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...
8,P3,P5,DISTINCT,OTHER:OTHER,OTHER:OTHER,"businessRole, operatingModel, deliveryModel, t...","partnerModel, valuePropositionThesis, offerThe...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
9,P4,P5,DISTINCT,OTHER:OTHER,OTHER:OTHER,"businessRole, operatingModel, deliveryModel, t...","partnerModel, valuePropositionThesis, offerThe...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.


## 18. Selected + Reserve Plans


In [19]:
selected_plans = plan_validation.acceptedPlans
reserve_plans = plan_validation.reservePlans
display(show_portfolio_plans(selected_plans + reserve_plans))
display({'selected': [p.planId for p in selected_plans], 'reserve': [p.planId for p in reserve_plans]})

,planId,제목,선택 상태,selectionScore,selectionReason,relationToPortfolio,Concept Family,Target Thesis,Use Context,Value Thesis,Offer Thesis,Solution Thesis,Architecture,비교 가치
0,P7,요리 초보를 위한 맞춤형 서비스,SELECTED,0.8373,Opportunity fit과 Concept clarity가 가장 높은 대표안,PORTFOLIO_SEED,기타 역할 · 기타 운영,요리할 시간이 적고 식재료 낭비를 줄이고 싶은 1~2인 가구,소량의 식재료를 필요로 하는 소비자에게 맞춤형 서비스를 제공하는 상황,요리 초보를 위한 맞춤형 식재료와 레시피를 제공하여 요리의 즐거움을 더한다.,요리 초보를 위한 맞춤형 식재료와 레시피 제공으로 요리의 즐거움을 더한다.,요리 초보를 위한 소량 식재료와 간편한 레시피를 통해 음식물 쓰레기를 줄인다.,"{'businessRole': 'OTHER', 'operatingModel': 'O...",요리 초보의 문제를 해결하기 위한 핵심 서비스.
1,P1,소량 맞춤형 식재료 서비스,SELECTED,0.8302,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,기타 역할 · 기타 운영,요리할 시간이 적고 식재료 낭비를 줄이고 싶은 1~2인 가구,소량의 식재료를 필요로 하는 소비자에게 맞춤형 서비스를 제공하는 상황,필요한 만큼의 식재료를 제공하여 음식물 쓰레기를 줄이고 요리를 쉽게 할 수 있도록 ...,소량의 식재료와 레시피를 제공하여 1~2인 가구의 요리 경험을 개선한다.,"개인 맞춤형 식재료와 레시피를 통해 요리의 번거로움을 줄이고, 음식물 쓰레기를 최소...","{'businessRole': 'OTHER', 'operatingModel': 'O...",1~2인 가구의 요리 문제를 해결하기 위한 핵심 서비스.
2,P3,간편 요리 키트 서비스,SELECTED,0.7430,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,기타 역할 · 기타 운영,요리할 시간이 적고 식재료 낭비를 줄이고 싶은 1~2인 가구,소량의 식재료를 필요로 하는 소비자에게 맞춤형 서비스를 제공하는 상황,간편하게 요리할 수 있는 키트를 제공하여 요리의 번거로움을 줄인다.,간편 요리 키트를 통해 요리의 즐거움을 제공한다.,소량의 식재료와 간편한 레시피를 통해 요리의 접근성을 높인다.,"{'businessRole': 'OTHER', 'operatingModel': 'O...",요리의 번거로움을 줄이고 1~2인 가구의 문제를 해결하는 서비스.
3,P4,맞춤형 식사 계획 서비스,SELECTED,0.6346,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,기타 역할 · 기타 운영,요리할 시간이 적고 식재료 낭비를 줄이고 싶은 1~2인 가구,소량의 식재료를 필요로 하는 소비자에게 맞춤형 서비스를 제공하는 상황,개인 맞춤형 식사 계획을 통해 요리의 효율성을 높인다.,맞춤형 식사 계획과 소량 식재료 제공으로 요리의 편리함을 더한다.,고객의 식사 스타일에 맞춘 식사 계획과 식재료 제공으로 음식물 쓰레기를 줄인다.,"{'businessRole': 'OTHER', 'operatingModel': 'O...",1~2인 가구의 식사 문제를 해결하기 위한 핵심 서비스.
4,P5,건강한 소량 식재료 서비스,SELECTED,0.5475,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,기타 역할 · 기타 운영,요리할 시간이 적고 식재료 낭비를 줄이고 싶은 1~2인 가구,소량의 식재료를 필요로 하는 소비자에게 맞춤형 서비스를 제공하는 상황,건강한 소량 식재료를 제공하여 요리의 질을 높인다.,건강한 식재료를 소량으로 제공하여 요리의 질을 높인다.,소량의 건강한 식재료를 통해 음식물 쓰레기를 줄인다.,"{'businessRole': 'OTHER', 'operatingModel': 'O...",건강한 식재료를 제공하여 1~2인 가구의 문제를 해결하는 서비스.
5,P2,레시피 기반 식재료 패키지,RESERVE,0.6939,Selected Portfolio 대비 marginal value가 낮아 reser...,"DISTINCT,VARIANT",기타 역할 · 기타 운영,요리할 시간이 적고 식재료 낭비를 줄이고 싶은 1~2인 가구,소량의 식재료를 필요로 하는 소비자에게 맞춤형 서비스를 제공하는 상황,레시피에 맞춘 소량의 식재료를 제공하여 요리의 편리함을 더한다.,레시피에 최적화된 식재료 패키지를 제공하여 요리의 효율성을 높인다.,고객의 요리 스타일에 맞춘 식재료 패키지를 통해 음식물 쓰레기를 줄인다.,"{'businessRole': 'OTHER', 'operatingModel': 'O...",요리의 편리함을 제공하여 1~2인 가구의 문제를 해결하는 서비스.
6,P6,신선한 재료 정기 배송 서비스,RESERVE,0.6825,Selected Portfolio 대비 marginal value가 낮아 reser...,"DISTINCT,VARIANT",기타 역할 · 기타 운영,요리할 시간이 적고 식재료 낭비를 줄이고 싶은 1~2인 가구,소량의 식재료를 필요로 하는 소비자에게 맞춤형 서비스를 제공하는 상황,신선한 식재료를 정기적으로 제공하여 요리의 질을 높인다.,신선한 식재료를 정기적으로 제공하여 요리의 질을 높인다.,정기 배송을 통해 신선한 식재료를 제공하여 음식물 쓰레기를 줄인다.,"{'businessRole': 'OTHER', 'operatingModel': 'O...",신선한 식재료를 제공하여 1~2인 가구의 문제를 해결하는 서비스.


{'selected': ['P7', 'P1', 'P3', 'P4', 'P5'], 'reserve': ['P2', 'P6']}

## 19. Candidate 1


In [20]:
candidate_one = await engine.expand_plan(seed, selected_plans[0], 1) if selected_plans else None
display(show_candidates([candidate_one]) if candidate_one else [])

,candidateId,lineageId,parentCandidateId,이름,핵심 작동방식,family,descriptor,수익,운영
0,C1,L1,None,요리 초보를 위한 맞춤형 서비스,요리 초보를 위한 소량 식재료와 간편한 레시피를 통해 음식물 쓰레기를 줄인다.,거래 중개 · 기타 운영,{'thesis': {'targetSegmentThesis': '요리할 시간이 적고...,구독 및 일회성 구매 옵션 제공,요리 초보 맞춤형 서비스 운영


## 20. Candidate 1 Korean/Governance


In [21]:
candidate_one_reports = []
display({'candidateId': candidate_one.candidateId if candidate_one else None,
         'status': 'PENDING_FULL_CANDIDATE_RECOVERY'})

{'candidateId': 'C1', 'status': 'PENDING_FULL_CANDIDATE_RECOVERY'}

## 21. Candidate 1 Actual Generic Descriptor


In [22]:
display(show_concept_descriptors([candidate_one]) if candidate_one else [])

,entityId,family,dimension,code,confidence,source
0,C1,INTERMEDIARY:OTHER,businessRole,INTERMEDIARY,HIGH,RULE
1,C1,INTERMEDIARY:OTHER,operatingModel,OTHER,LOW,UNKNOWN
2,C1,INTERMEDIARY:OTHER,partnerModel,OTHER,LOW,UNKNOWN
3,C1,INTERMEDIARY:OTHER,deliveryModel,OTHER,LOW,UNKNOWN
4,C1,INTERMEDIARY:OTHER,transactionModel,OTHER,LOW,UNKNOWN
5,C1,INTERMEDIARY:OTHER,monetizationModel,OTHER,LOW,UNKNOWN
6,C1,INTERMEDIARY:OTHER,customerInteractionModel,OTHER,LOW,UNKNOWN
7,C1,INTERMEDIARY:OTHER,dataDependency,NONE,NaN,NaN
8,C1,INTERMEDIARY:OTHER,physicalDependency,NONE,NaN,NaN


## 22. Candidate 1 Fidelity


In [23]:
display({'candidateId': candidate_one.candidateId if candidate_one else None,
         'fidelity': '전체 Candidate Recovery 단계에서 semantic fallback 포함 검증'})

{'candidateId': 'C1',
 'fidelity': '전체 Candidate Recovery 단계에서 semantic fallback 포함 검증'}

## 23. Remaining Candidates


In [24]:
remaining_candidates = []
for i, plan in enumerate(selected_plans[1:], 2):
    remaining_candidates.append(await engine.expand_plan(seed, plan, i))
candidate_drafts = ([candidate_one] if candidate_one else []) + remaining_candidates
display(show_candidates(candidate_drafts))

,candidateId,lineageId,parentCandidateId,이름,핵심 작동방식,family,descriptor,수익,운영
0,C1,L1,None,요리 초보를 위한 맞춤형 서비스,요리 초보를 위한 소량 식재료와 간편한 레시피를 통해 음식물 쓰레기를 줄인다.,거래 중개 · 기타 운영,{'thesis': {'targetSegmentThesis': '요리할 시간이 적고...,구독 및 일회성 구매 옵션 제공,요리 초보 맞춤형 서비스 운영
1,C2,L2,None,소량 맞춤형 식재료 서비스,"개인 맞춤형 식재료와 레시피를 통해 요리의 번거로움을 줄이고, 음식물 쓰레기를 최소...",기타 역할 · 기타 운영,{'thesis': {'targetSegmentThesis': '요리할 시간이 적고...,구독 모델 및 일회성 구매 옵션 제공,고객 맞춤형 주문 시스템 운영
2,C3,L3,None,간편 요리 키트 서비스,주문 후 1~2일 이내 배송되는 간편 요리 키트를 통해 요리의 즐거움을 제공한다.,기타 역할 · 기타 운영,{'thesis': {'targetSegmentThesis': '요리할 시간이 적고...,구독 및 일회성 구매 옵션 제공,고객 맞춤형 요리 키트 운영
3,C4,L4,None,맞춤형 식사 계획 서비스,고객의 식사 스타일에 맞춘 식사 계획과 식재료 제공으로 음식물 쓰레기를 줄인다.,기타 역할 · 기타 운영,{'thesis': {'targetSegmentThesis': '요리할 시간이 적고...,구독 및 일회성 구매 옵션 제공,고객 맞춤형 식사 계획 운영
4,C5,L5,None,건강한 소량 식재료 서비스,소량의 건강한 식재료를 통해 음식물 쓰레기를 줄인다.,거래 중개 · 기타 운영,{'thesis': {'targetSegmentThesis': '요리할 시간이 적고...,구독 및 일회성 구매 옵션 제공,건강한 식재료 패키지 운영


## 24. Candidate Actual Generic Descriptors


In [25]:
display(show_concept_descriptors(candidate_drafts))

,entityId,family,dimension,code,confidence,source
0,C1,INTERMEDIARY:OTHER,businessRole,INTERMEDIARY,HIGH,RULE
1,C1,INTERMEDIARY:OTHER,operatingModel,OTHER,LOW,UNKNOWN
2,C1,INTERMEDIARY:OTHER,partnerModel,OTHER,LOW,UNKNOWN
3,C1,INTERMEDIARY:OTHER,deliveryModel,OTHER,LOW,UNKNOWN
4,C1,INTERMEDIARY:OTHER,transactionModel,OTHER,LOW,UNKNOWN
5,C1,INTERMEDIARY:OTHER,monetizationModel,OTHER,LOW,UNKNOWN
6,C1,INTERMEDIARY:OTHER,customerInteractionModel,OTHER,LOW,UNKNOWN
7,C1,INTERMEDIARY:OTHER,dataDependency,NONE,NaN,NaN
8,C1,INTERMEDIARY:OTHER,physicalDependency,NONE,NaN,NaN
9,C2,OTHER:OTHER,businessRole,OTHER,LOW,UNKNOWN


## 25. Candidate Recovery / Portfolio Relations


In [26]:
candidate_preparation = await engine.prepare_candidate_portfolio(
    seed, plan_validation, max_concepts=MAX_CONCEPTS, initial_candidates=candidate_drafts)
candidates = candidate_preparation.candidates
candidate_reports = candidate_preparation.reports
display(show_candidate_recovery(candidate_preparation))
candidate_pairwise = [engine.compare_candidates(candidates[i], candidates[j])
                      for i in range(len(candidates)) for j in range(i + 1, len(candidates))]
display(show_plan_diversity(candidate_pairwise))

{'summary':    candidateGenerated  candidateAcceptedInitially  candidateRegenerated  \
 0                   5                           5                     0   
 
    candidateRecovered  reservePlansActivated  candidateRecoveryReplans  \
 0                   0                      0                         0   
 
    finalCandidatePortfolio  
 0                        5  ,
 'attempts':   candidateId  schemaValid  hardLockPreserved  semanticAnchorPreserved  \
 0          C1         True               True                     True   
 1          C2         True               True                     True   
 2          C3         True               True                     True   
 3          C4         True               True                     True   
 4          C5         True               True                     True   
 
    planFidelity anchorDecision fidelityDecision  contentLanguageValid  \
 0          True           PASS          ADAPTED                  True   
 1        

,A,B,판정,Family A,Family B,겹침,실질 차이,단계,semantic judge,관계 설명
0,C1,C2,DISTINCT,INTERMEDIARY:OTHER,OTHER:OTHER,"operatingModel, transactionModel, monetization...","businessRole, partnerModel, deliveryModel, val...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
1,C1,C3,DISTINCT,INTERMEDIARY:OTHER,OTHER:OTHER,"operatingModel, partnerModel, deliveryModel, t...","businessRole, valuePropositionThesis, offerThe...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
2,C1,C4,DISTINCT,INTERMEDIARY:OTHER,OTHER:OTHER,"operatingModel, partnerModel, deliveryModel, t...","businessRole, valuePropositionThesis, offerThe...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
3,C1,C5,DISTINCT,INTERMEDIARY:OTHER,INTERMEDIARY:OTHER,"businessRole, operatingModel, deliveryModel, t...","partnerModel, valuePropositionThesis, offerThe...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
4,C2,C3,DISTINCT,OTHER:OTHER,OTHER:OTHER,"businessRole, operatingModel, transactionModel...","partnerModel, deliveryModel, valuePropositionT...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
5,C2,C4,DISTINCT,OTHER:OTHER,OTHER:OTHER,"businessRole, operatingModel, transactionModel...","partnerModel, deliveryModel, valuePropositionT...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
6,C2,C5,DISTINCT,OTHER:OTHER,INTERMEDIARY:OTHER,"operatingModel, partnerModel, transactionModel...","businessRole, deliveryModel, valuePropositionT...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
7,C3,C4,DISTINCT,OTHER:OTHER,OTHER:OTHER,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, offerThesis, solutionT...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
8,C3,C5,DISTINCT,OTHER:OTHER,INTERMEDIARY:OTHER,"operatingModel, deliveryModel, transactionMode...","businessRole, partnerModel, valuePropositionTh...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
9,C4,C5,DISTINCT,OTHER:OTHER,INTERMEDIARY:OTHER,"operatingModel, deliveryModel, transactionMode...","businessRole, partnerModel, valuePropositionTh...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.


## 26. Legal Fact Completeness + Business Design Completion + C1 Fact Pattern


In [27]:
prechecks = [engine.legal_precheck(item) for item in candidates]
display(show_legal_precheck(prechecks))
legal_preparation = await engine.prepare_legal_candidates(seed, candidates)
candidates_before_legal = candidates
candidates = legal_preparation.candidates
display({'factCompleteness': [item.model_dump(mode='json') for item in legal_preparation.reports],
         'completionAttempted': legal_preparation.completionAttempted,
         'completionValidated': legal_preparation.completionValidated,
         'completionAccepted': legal_preparation.completionAccepted,
         'completionExhausted': legal_preparation.completionExhausted,
         'excludedCandidates': legal_preparation.excludedCandidates})
display(show_legal_fact_pattern(candidates[0].candidate, seed) if candidates else [])

,candidateId,label,directSeller,intermediary,regulatedPhysicalActivity,personalDataDependency,qualificationDependency,riskHints
0,C1,Structural risk precheck — not final legal review,True,True,False,False,False,[]
1,C2,Structural risk precheck — not final legal review,True,False,False,False,False,[]
2,C3,Structural risk precheck — not final legal review,False,False,False,False,False,[]
3,C4,Structural risk precheck — not final legal review,True,False,False,False,False,[]
4,C5,Structural risk precheck — not final legal review,True,True,False,False,False,[]


,field,legalFact,source,authority,decision
0,platformRole,소비자와 공급자를 연결하는 플랫폼 역할,CONCEPT_GENERATED,REVIEWABLE,PROPOSED
1,providerRole,식재료 및 레시피 제공자,CONCEPT_GENERATED,REVIEWABLE,PROPOSED
2,sellerRole,소비자에게 판매하는 역할,CONCEPT_GENERATED,REVIEWABLE,PROPOSED
3,intermediaryRole,소비자와 공급자 간의 중개 역할,CONCEPT_GENERATED,REVIEWABLE,PROPOSED
4,transactionFlow,"[소비자가 앱을 통해 주문, 주문 후 1~2일 이내 배송]",CONCEPT_GENERATED,REVIEWABLE,PROPOSED
5,paymentFlow,"[구독 결제, 일회성 구매 결제]",CONCEPT_GENERATED,REVIEWABLE,PROPOSED
6,personalDataUsage,[],CONCEPT_GENERATED,REVIEWABLE,PROPOSED
7,physicalActivities,[],CONCEPT_GENERATED,REVIEWABLE,PROPOSED
8,partnerRequirements,[],CONCEPT_GENERATED,REVIEWABLE,PROPOSED
9,qualificationRequirements,[],CONCEPT_GENERATED,REVIEWABLE,PROPOSED


## 27. Prepared Legal C1 Evidence Summary


In [28]:
legal_adapter = CurrentLegalAdapter()
legal_c1_input = legal_adapter.task_input(candidates[0].candidate, seed) if candidates else None
display({'candidateId': candidates[0].candidateId if candidates else None,
         'externalFacts': legal_c1_input['externalFactContext']['facts'] if legal_c1_input else [],
         'note': '공식 근거 수와 allowed index는 Full Legal 응답/실패 diagnostics에서 확인'})

{'candidateId': 'C1',
 'externalFacts': [],
 'note': '공식 근거 수와 allowed index는 Full Legal 응답/실패 diagnostics에서 확인'}

## 28. Full Evidence Judgment — C1 Staged Smoke


In [29]:
RUN_FULL_LEGAL_C1 = True  # staged LIVE 확인 후 True
legal_one = None
if RUN_FULL_LEGAL_C1 and candidates:
    try:
        legal_one = await engine.review_legal_candidate(seed, candidates[0])
        display(show_legal_result([legal_one]))
    except Exception:
        display(show_legal_failure(candidates[0].candidateId, engine.gateway))
else:
    print('SKIPPED — RUN_FULL_LEGAL_C1=True로 명시해야 실행됩니다.')

,candidateId,route,productionStatus,source,safeSummary,requiredControls,requiredPartnersAndQualifications,requiredDisclosures,prohibitedVariants,evidenceCount,evidenceRefs,evidenceDiagnostics
0,C1,REDESIGN_WITHIN_LINEAGE,REDESIGNABLE,OFFICIAL_EVIDENCE,법률검토 전에 Concept의 사업 구조를 보완해야 합니다.,[],[],[],[],16,"[{'referenceIndex': 0, 'sourceType': 'OFFICIAL...",{}


## 29. C1 Route + Staged Redesign/Compliance/Second Legal


In [30]:
c1_portfolio, c1_legal_all, c1_required_inputs, c1_redesigned, c1_replanned = ([], [], [], 0, 0)
if legal_one and candidates:
    c1_portfolio, c1_legal_all, c1_required_inputs, c1_redesigned, c1_replanned = await engine.resolve_legal(
        seed, candidate_preparation.usedPlans, candidates[:1], [legal_one])
display({'initialRoute': legal_one.route.value if legal_one else 'SKIPPED',
         'redesignRequirements': legal_one.redesignRequirements if legal_one else [],
         'recoveryReviews': [item.model_dump(mode='json') for item in c1_legal_all[1:]],
         'requiredInputs': c1_required_inputs, 'terminalCandidates': len(c1_portfolio),
         'redesigned': c1_redesigned, 'replanned': c1_replanned})

,candidateId,route,productionStatus,source,safeSummary,requiredControls,requiredPartnersAndQualifications,requiredDisclosures,prohibitedVariants,evidenceCount,evidenceRefs,evidenceDiagnostics
0,C2,REDESIGN_WITHIN_LINEAGE,REDESIGNABLE,OFFICIAL_EVIDENCE,법률검토 전에 Concept의 사업 구조를 보완해야 합니다.,[],[],[],[],16,"[{'referenceIndex': 0, 'sourceType': 'OFFICIAL...",{}
1,C3,REDESIGN_WITHIN_LINEAGE,REDESIGNABLE,OFFICIAL_EVIDENCE,법률검토 전에 Concept의 사업 구조를 보완해야 합니다.,[],[],[],[],11,"[{'referenceIndex': 0, 'sourceType': 'OFFICIAL...",{}
2,C4,REDESIGN_WITHIN_LINEAGE,REDESIGNABLE,OFFICIAL_EVIDENCE,법률검토 전에 Concept의 사업 구조를 보완해야 합니다.,[],[],[],[],16,"[{'referenceIndex': 0, 'sourceType': 'OFFICIAL...",{}
3,C5,REDESIGN_WITHIN_LINEAGE,REDESIGNABLE,OFFICIAL_EVIDENCE,법률검토 전에 Concept의 사업 구조를 보완해야 합니다.,[],[],[],[],20,"[{'referenceIndex': 0, 'sourceType': 'OFFICIAL...",{}


{'Plan Selected': 5,
 'Candidate Generated': 5,
 'Candidate Accepted': 5,
 'Legal Reviewed': 5}

## 30. Remaining 4 Legal + Exhaustive Recovery Summary


In [31]:
RUN_REMAINING_LEGAL = True
legal_remaining = []
portfolio, legal_all, required_inputs = (list(c1_portfolio), list(c1_legal_all), list(c1_required_inputs))
redesigned_count, replanned_count = c1_redesigned, c1_replanned
c1_terminal = bool(c1_portfolio or c1_required_inputs or (c1_legal_all and c1_legal_all[-1].route.value == 'SYSTEM_FAILURE'))
if RUN_REMAINING_LEGAL and c1_terminal and len(candidates) > 1:
    legal_remaining = await engine.review_legal(seed, candidates[1:])
    rest_portfolio, rest_legal, rest_inputs, rest_redesigned, rest_replanned = await engine.resolve_legal(
        seed, candidate_preparation.usedPlans, candidates[1:], legal_remaining)
    portfolio += rest_portfolio; legal_all += rest_legal; required_inputs += rest_inputs
    redesigned_count += rest_redesigned; replanned_count += rest_replanned
else:
    print('SKIPPED — C1이 정상 terminal에 도달한 후 remaining Legal을 실행합니다.')
legal_initial = ([legal_one] if legal_one else []) + legal_remaining
display({'recoveryTrace': [item.model_dump(mode='json') for item in legal_all
                           if item.candidateId not in {x.candidateId for x in legal_initial}],
         'requiredInputs': required_inputs, 'metrics': engine._legal_metrics})
print({'Plan Selected': len(selected_plans),
       'Candidate Generated': candidate_preparation.candidateGenerated,
       'Candidate Accepted': len(candidates),
       'Legal Accepted': sum(item.route.value == 'ACCEPT' for item in legal_all),
       'Legal Redesigned': redesigned_count, 'Legal Replanned': replanned_count,
       'Final Portfolio': len(portfolio)})

{'Plan Selected': 5, 'Candidate Generated': 5, 'Candidate Accepted': 5, 'Legal Accepted': 0, 'Legal Redesigned': 0, 'Legal Replanned': 0, 'Final Portfolio': 0}


## 31. Replan


In [32]:
display(show_replan(type('PortfolioView', (), {'concepts': portfolio})()))
print({'replanned': replanned_count, 'reserveAvailable': len(reserve_plans)})

""


{'replanned': 0, 'reserveAvailable': 2}


## 32. Final Portfolio


In [33]:
legal_terminal_status = ('READY' if portfolio else
    'LEGAL_RECOVERY_COMPLETE_NO_ACCEPTED_CANDIDATE' if legal_initial and len(legal_initial) == len(candidates)
    else 'LEGAL_PENDING')
display(show_final_portfolio(type('PortfolioView', (), {'concepts': portfolio})()) if portfolio else {'status': legal_terminal_status})

{'status': 'LEGAL_PENDING'}

## 33. Unresolved Candidate Summary


In [34]:
display(required_inputs if required_inputs else {'unresolved': []})

{'unresolved': []}

## 34. Manual Concept Selection


In [35]:
SELECTED_CANDIDATE_ID = portfolio[0].candidateId if portfolio else None  # 사용자가 수정
selected_concept = next((item for item in portfolio if item.candidateId == SELECTED_CANDIDATE_ID), None)
print({'selectedCandidateId': SELECTED_CANDIDATE_ID})

{'selectedCandidateId': None}


## 35. 7 Hypotheses


In [36]:
hypotheses = engine.build_or_load_current_hypothesis_contract(selected_concept) if selected_concept else []
display(show_hypotheses(hypotheses))

""


## 36. Confirm / Edit


In [37]:
CONFIRM_ALL_PROPOSED = True
HYPOTHESIS_EDITS = {
    # 'PRICE': '월 17,900원',
}
confirmed_hypotheses = engine.confirm_hypotheses(
    hypotheses, HYPOTHESIS_EDITS, confirm_all_proposed=CONFIRM_ALL_PROPOSED) if hypotheses else []
display(show_hypotheses(confirmed_hypotheses))

""


## 37. Actual Delta Legal


In [38]:
RUN_DELTA_LEGAL = True
delta_legal_result = None
if RUN_DELTA_LEGAL and selected_concept and any(h.deltaLegalRequired for h in confirmed_hypotheses):
    delta_legal_result = await engine.review_delta_legal(seed, selected_concept, confirmed_hypotheses)
    confirmed_hypotheses = engine.mark_delta_legal_reviewed(confirmed_hypotheses, delta_legal_result)
display(delta_legal_result.model_dump(mode='json') if delta_legal_result else {'status': 'NOT_REQUIRED_OR_SKIPPED'})

{'status': 'NOT_REQUIRED_OR_SKIPPED'}

## 38. Market Seed


In [39]:
handoff = None
if selected_concept and legal_all:
    handoff = engine.build_downstream_handoff(seed, selected_concept, confirmed_hypotheses, legal_all)
display(handoff.marketAnalysisSeedSnapshot if handoff else {'status': 'NOT_READY'})

{'status': 'NOT_READY'}

## 39. Marketing Source


In [40]:
display(handoff.marketingSourceSnapshot if handoff else {'status': 'NOT_READY'})

{'status': 'NOT_READY'}

## 40. Contract Compatibility


In [41]:
display(show_downstream_handoff(handoff) if handoff else {'contract': 'NOT_READY'})

{'contract': 'NOT_READY'}

## 41. Trace


In [42]:
display(show_trace(engine.trace))

,순서,시각,stage,action,entity,parent,status,mode,호출,요약,decision,reasonCode
0,1,2026-08-10T00:12:27.630898+00:00,CREATED,CREATED,NaN,NaN,RUNNING,LIVE,NaN,V2 Lab 실행을 생성했습니다.,NaN,NaN
1,2,2026-08-10T00:12:32.438772+00:00,SAFETY_CHECKING,IDEA_BRIEF_DERIVED,NaN,NaN,PASS,LIVE,1.0,Idea interpretation/readiness를 보존했습니다: READY_F...,NaN,NaN
2,3,2026-08-10T00:12:32.438796+00:00,SAFETY_CHECKING,READINESS_INCONSISTENT,NaN,NaN,WARNING,LIVE,NaN,READY_FOR_REVIEW이지만 score=0입니다. V2 gating은 막지 ...,READINESS_INCONSISTENT,NaN
3,4,2026-08-10T00:12:32.508904+00:00,SEED_ANALYZING,ANALYZED,lab-idea-brief,NaN,PASS,LIVE,NaN,필수 3개와 LOCK 3개를 분류했습니다.,NaN,NaN
4,5,2026-08-10T00:12:32.509154+00:00,SEED_ANALYZING,DESIGN_SPACE_READY,NaN,NaN,PASS,LIVE,NaN,Open=11 Constrained=0 Breadth=EXPLORE,NaN,NaN
5,6,2026-08-10T00:12:32.568279+00:00,PLANNING,STARTED,NaN,NaN,RUNNING,LIVE,NaN,최대 5개 동적 plan을 요청합니다.,NaN,NaN
6,7,2026-08-10T00:12:58.652230+00:00,PLANNING,DRAFTS_GENERATED,NaN,NaN,PASS,LIVE,2.0,Plan draft pool=7,NaN,NaN
7,8,2026-08-10T00:12:58.654566+00:00,PLANNING,NORMALIZED,NaN,NaN,PASS,LIVE,NaN,System metadata를 부여한 Plan=7,NaN,NaN
8,9,2026-08-10T00:13:06.555897+00:00,PLANNING,ARCHITECTURE_SEMANTIC_FALLBACK,NaN,NaN,PASS,LIVE,3.0,low-confidence Plan architecture 7개를 batch 분류했...,NaN,NaN
9,10,2026-08-10T00:13:06.555928+00:00,PLAN_VALIDATING,STARTED,NaN,NaN,RUNNING,LIVE,NaN,Opportunity·LOCK·DUPLICATE/VARIANT/DISTINCT 관계...,NaN,NaN


## 42. Provider/Legal Usage


In [43]:
display(show_provider_usage(engine.gateway.usage))
print('상위 외부 작업 수는 내부 AI/MOLEG 네트워크 호출 수와 동일하다고 주장하지 않습니다.')

,논리 작업,상위 외부 작업,논리 stage별,상위 외부 작업 stage별,재시도,소요(ms),모드별,token,보고 비용
0,35,35,"{'SAFETY_CHECKING': 1, 'PLANNING': 1, 'NORMALI...","{'SAFETY_CHECKING': 1, 'PLANNING': 1, 'NORMALI...",0,304806,{'LIVE': 35},None,None


상위 외부 작업 수는 내부 AI/MOLEG 네트워크 호출 수와 동일하다고 주장하지 않습니다.


## 43. Replay Manifest


In [44]:
display(show_replay_manifest(engine.gateway))

{'status': 'REPLAY_PARTIAL',
 'entries':                   operation  \
 0                 PLAN_POOL   
 1         SEMANTIC_RELATION   
 2     IDEA_BRIEF_DERIVATION   
 3              LEGAL_REVIEW   
 4              LEGAL_REVIEW   
 5                    EXPAND   
 6              LEGAL_REVIEW   
 7   NORMALIZE_ARCHITECTURES   
 8   NORMALIZE_ARCHITECTURES   
 9                    EXPAND   
 10                   EXPAND   
 11             LEGAL_REVIEW   
 12             LEGAL_REVIEW   
 13  NORMALIZE_ARCHITECTURES   
 14                   EXPAND   
 15                   EXPAND   
 16                   EXPAND   
 17  NORMALIZE_ARCHITECTURES   
 18                   EXPAND   
 19  NORMALIZE_ARCHITECTURES   
 20        SEMANTIC_RELATION   
 21  NORMALIZE_ARCHITECTURES   
 22  NORMALIZE_ARCHITECTURES   
 23             LEGAL_REVIEW   
 24                   EXPAND   
 25                   EXPAND   
 26        SEMANTIC_RELATION   
 27                 REDESIGN   
 28             LEGAL_REVIEW   


## 44. One-click MOCK


In [45]:
mock_result = await ConceptPortfolioEngine('MOCK').run_full(
    TEST_INPUT, max_concepts=MAX_CONCEPTS, auto_confirm_hypotheses=True)
display(show_run_summary(mock_result))
assert mock_result.handoff and mock_result.handoff.contractStatus == 'CONTRACT_PASS'

,safety,requestedMaximum,planned,planDuplicatesRemoved,candidatesExpanded,candidateGenerated,candidateAcceptedInitially,candidateRegenerated,candidateRecovered,reservePlansActivated,candidateRecoveryReplans,legalAccepted,legalRedesigned,replanned,finalPortfolio,portfolioStatus,selectedConcept,downstreamHandoff,providerCalls,totalDurationMs
0,PASS,5,7,0,5,5,5,0,0,0,0,5,0,0,5,READY_FULL,직접 운영 핵심형,PASS,0,52


## 45. One-click REPLAY


In [46]:
RUN_ONE_CLICK_REPLAY = False
replay_result = None
if RUN_ONE_CLICK_REPLAY:
    replay_gateway = ProviderGateway('REPLAY', recordings_dir=RECORDINGS_DIR)
    replay_result = await ConceptPortfolioEngine('REPLAY', gateway=replay_gateway).run_full(
        TEST_INPUT, max_concepts=MAX_CONCEPTS, auto_confirm_hypotheses=False)
display(show_run_summary(replay_result) if replay_result else {'status': 'SKIPPED'})

{'status': 'SKIPPED'}

## 46. One-click LIVE


In [47]:
RUN_ONE_CLICK_LIVE = False
live_result = None
if RUN_ONE_CLICK_LIVE:
    assert MODE == 'LIVE', 'MODE=LIVE를 먼저 명시하세요.'
    assert RUN_FULL_LEGAL_C1 and RUN_REMAINING_LEGAL, 'staged LIVE Legal 확인이 먼저입니다.'
    live_result = await engine.run_full(TEST_INPUT, max_concepts=MAX_CONCEPTS,
                                        auto_confirm_hypotheses=False)
display(show_run_summary(live_result) if live_result else {'status': 'SKIPPED'})

{'status': 'SKIPPED'}